In [39]:
cd "C:\Users\Lenovo\Desktop\github projects\telcos customer churn"

C:\Users\Lenovo\Desktop\github projects\telcos customer churn


# Feature selection .py file 

In [ ]:
%%writefile src/feature_selection.py

import os
import sys
from dataclasses import dataclass

import pandas as pd 
import numpy as np

from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_classif

from src.exception import CustomException
from src.logger import logging
from src.utils import save_object


@dataclass
class FeatureSelectionConfig:
    train_data_path = os.path.join(
        "data",
        "processed",
        "train.csv"
    )

    report_path = os.path.join(
        "reports",
        "feature_selection_report.csv"
    )

    summary_report_path = os.path.join(
        "reports",
        "feature_selection_summary.csv"
    )

    selected_features_path = os.path.join(
        "artifacts",
        "selected_features.pkl"
    )

    feature_names_path = os.path.join(
        "artifacts",
        "feature_names.pkl"
    )


class FeatureSelection:
    def __init__(self):
        self.config = FeatureSelectionConfig()

    def initiate_feature_selection(self):
        try:
            logging.info("Loading Processesd Training Dataset")
            train_df=pd.read_csv(self.config.train_data_path)

            X=train_df.iloc[:,:-1]
            y=train_df.iloc[:,-1]

            logging.info(f"Original Feature Count: {X.shape[1]}")

            #Variance Tersold

            selector = VarianceThreshold(threshold=0.0)

            selector.fit(X)

            selected_columns = X.columns[selector.get_support()]

            removed_variance_features = X.columns[~selector.get_support()].tolist()

            X = X[selected_columns]

            logging.info(f"Removed {len(removed_variance_features)} low variance features.")

            # corelation filter

            corr_matrix=X.corr().abs()
            upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

            columns_to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

            removed_correlation_features = columns_to_drop.copy()

            X = X.drop(columns=columns_to_drop)

            logging.info(f"Removed {len(columns_to_drop)} highly correlated features.")

            # Step 3 : Mutual Information

            mi_scores= mutual_info_classif(X,y,random_state=42)
            mi_df = pd.DataFrame({"Feature": X.columns,"MutualInformation": mi_scores})

            mi_df["Rank"] = (mi_df["MutualInformation"].rank(ascending=False, method="dense").astype(int))

            mi_df = mi_df.sort_values(by="MutualInformation",ascending=False).reset_index(drop=True)

            #report save 


            os.makedirs("reports", exist_ok=True)

            mi_df.to_csv(
                self.config.report_path,
                index=False
            )

            logging.info("Feature selection report saved.")

            # --------------------------------------------------
            # Save Feature List
            # --------------------------------------------------

            selected_feature_list = mi_df["Feature"].tolist()

            feature_names = X.columns.tolist()

            save_object(file_path=self.config.feature_names_path,obj=feature_names)

            logging.info("Feature names saved successfully.")

            os.makedirs("artifacts", exist_ok=True)

            save_object(file_path=self.config.selected_features_path, obj=selected_feature_list)

            logging.info("Selected feature list saved.")


            summary_df = pd.DataFrame({"Stage": ["Original Features","Removed by Variance Threshold","Removed by Correlation","Final Selected Features"],
                       "Count": [train_df.shape[1] - 1,len(removed_variance_features),len(removed_correlation_features),len(selected_feature_list)]})

            summary_df.to_csv(self.config.summary_report_path,index=False)

            logging.info("Feature selection summary report saved.")

            return mi_df, summary_df

        except Exception as e:
            raise CustomException(e, sys)


In [40]:
import importlib
import src.feature_selection

importlib.reload(src.feature_selection)

from src.feature_selection import FeatureSelection

In [41]:
import os
import pandas as pd

from src.feature_selection import FeatureSelection

In [42]:
selector = FeatureSelection()

feature_report, summary_report = selector.initiate_feature_selection()

In [43]:
summary_report

,Stage,Count
0,Original Features,58
1,Removed by Variance Threshold,0
2,Removed by Correlation,13
3,Final Selected Features,45


In [44]:
feature_report.head(10)

,Feature,MutualInformation,Rank
0,num_pipeline__ContractRisk,0.094044,1
1,cat_pipeline__Contract_Month-to-month,0.085070,2
2,num_pipeline__MonthlyChargePerService,0.080955,3
3,num_pipeline__tenure,0.076660,4
4,cat_pipeline__TechSupport_No,0.061098,5
5,cat_pipeline__Contract_Two year,0.057809,6
6,cat_pipeline__TenureGroup_New,0.056373,7
7,cat_pipeline__OnlineSecurity_No,0.055367,8
8,num_pipeline__MonthlyCharges,0.051786,9
9,cat_pipeline__InternetService_Fiber optic,0.046129,10


In [45]:
print("Feature Report Shape:", feature_report.shape)

Feature Report Shape: (45, 3)


In [46]:
print("Feature Selection Report Exists:",
      os.path.exists("reports/feature_selection_report.csv"))

print("Summary Report Exists:",
      os.path.exists("reports/feature_selection_summary.csv"))

Feature Selection Report Exists: True
Summary Report Exists: True


In [47]:
print("Selected Features PKL Exists:",
      os.path.exists("artifacts/selected_features.pkl"))

print("Feature Names PKL Exists:",
      os.path.exists("artifacts/feature_names.pkl"))

Selected Features PKL Exists: True
Feature Names PKL Exists: True


In [48]:
saved_report = pd.read_csv("reports/feature_selection_report.csv")

saved_report.head()

,Feature,MutualInformation,Rank
0,num_pipeline__ContractRisk,0.094044,1
1,cat_pipeline__Contract_Month-to-month,0.085070,2
2,num_pipeline__MonthlyChargePerService,0.080955,3
3,num_pipeline__tenure,0.076660,4
4,cat_pipeline__TechSupport_No,0.061098,5


In [49]:
saved_summary = pd.read_csv("reports/feature_selection_summary.csv")

saved_summary

,Stage,Count
0,Original Features,58
1,Removed by Variance Threshold,0
2,Removed by Correlation,13
3,Final Selected Features,45


In [50]:
print("=" * 50)
print("PHASE 4.2 FEATURE SELECTION VERIFICATION")
print("=" * 50)

print(f"Original Train Dataset Shape : {pd.read_csv('data/processed/train.csv').shape}")
print(f"Selected Feature Count       : {len(feature_report)}")
print(f"Top Feature                  : {feature_report.iloc[0]['Feature']}")

print("\nArtifacts Created Successfully:")
print("✔ reports/feature_selection_report.csv")
print("✔ reports/feature_selection_summary.csv")
print("✔ artifacts/selected_features.pkl")
print("✔ artifacts/feature_names.pkl")

print("\nPHASE 4.2 COMPLETED SUCCESSFULLY")

PHASE 4.2 FEATURE SELECTION VERIFICATION
Original Train Dataset Shape : (5634, 59)
Selected Feature Count       : 45
Top Feature                  : num_pipeline__ContractRisk

Artifacts Created Successfully:
✔ reports/feature_selection_report.csv
✔ reports/feature_selection_summary.csv
✔ artifacts/selected_features.pkl
✔ artifacts/feature_names.pkl

PHASE 4.2 COMPLETED SUCCESSFULLY


In [51]:
## Feature Finalisation 

In [52]:
%%writefile src/feature_finalization.py

import os
import sys
from dataclasses import dataclass

import pandas as pd

from src.exception import CustomException
from src.logger import logging
from src.utils import load_object


# ==========================================================
# Configuration
# ==========================================================

@dataclass
class FeatureFinalizationConfig:

    train_data_path = os.path.join(
        "data", "processed", "train.csv"
    )

    test_data_path = os.path.join(
        "data", "processed", "test.csv"
    )

    selected_features_path = os.path.join(
        "artifacts", "selected_features.pkl"
    )

    final_train_path = os.path.join(
        "data", "final", "train_selected.csv"
    )

    final_test_path = os.path.join(
        "data", "final", "test_selected.csv"
    )


class FeatureFinalization:

    def __init__(self):
        self.config = FeatureFinalizationConfig()

    def initiate_feature_finalization(self):

        try:
            logging.info("Loading processed train and test datasets.")

            train_df= pd.read_csv(self.config.train_data_path)
            test_df= pd.read_csv(self.config.test_data_path)

            logging.info("Loading selected feature list.")

            selected_features=load_object(self.config. selected_features_path)

            train_selected= train_df[selected_features + [train_df.columns[-1]]]
            test_selected=test_df[selected_features + [test_df.columns[-1]]]



            os.makedirs(
                os.path.dirname(self.config.final_train_path),
                exist_ok=True
            )

            train_selected.to_csv(
                self.config.final_train_path,
                index=False
            )

            test_selected.to_csv(
                self.config.final_test_path,
                index=False
            )

            logging.info("Final datasets created successfully.")

            return train_selected, test_selected

        except Exception as e:
            raise CustomException(e, sys)

Overwriting src/feature_finalization.py


In [53]:
import os
import pandas as pd

from src.feature_finalization import FeatureFinalization

In [54]:
finalizer = FeatureFinalization()

train_selected, test_selected = finalizer.initiate_feature_finalization()

In [55]:
print("Train Selected Shape :", train_selected.shape)
print("Test Selected Shape  :", test_selected.shape)

Train Selected Shape : (5634, 46)
Test Selected Shape  : (1409, 46)


In [56]:
print("Columns Match :",train_selected.columns.tolist() == test_selected.columns.tolist())

Columns Match : True


In [57]:
print(os.path.exists("data/final/train_selected.csv"))
print(os.path.exists("data/final/test_selected.csv"))

True
True


In [58]:
print("=" * 50)
print("PHASE 4.3 VERIFICATION")
print("=" * 50)

print("Original Train :", pd.read_csv("data/processed/train.csv").shape)
print("Final Train    :", train_selected.shape)

print("Original Test  :", pd.read_csv("data/processed/test.csv").shape)
print("Final Test     :", test_selected.shape)

print("\nFiles Created:")
print("✔ data/final/train_selected.csv")
print("✔ data/final/test_selected.csv")

print("\nPHASE 4 COMPLETED SUCCESSFULLY")

PHASE 4.3 VERIFICATION
Original Train : (5634, 59)
Final Train    : (5634, 46)
Original Test  : (1409, 59)
Final Test     : (1409, 46)

Files Created:
✔ data/final/train_selected.csv
✔ data/final/test_selected.csv

PHASE 4 COMPLETED SUCCESSFULLY
